# OpenAI API key smoke test

This notebook reads `OPENAI_API_KEY` directly from this project's `backend/.env`; it does **not** display or save the key. A successful response proves that the same key used by the backend can authenticate and make a basic model request.

In [ ]:
%pip install -q openai

from pathlib import Path
from openai import OpenAI, APIStatusError

env_path = Path('/Users/yanliu/Projects/Manipulation research-platform/backend/.env')
if not env_path.is_file():
    raise FileNotFoundError(f'Cannot find backend environment file: {env_path}')

api_key = ''
for line in env_path.read_text().splitlines():
    stripped = line.strip()
    if stripped.startswith('OPENAI_API_KEY='):
        api_key = stripped.split('=', 1)[1].strip().strip(chr(34)).strip(chr(39))
        break

if not api_key:
    raise RuntimeError('OPENAI_API_KEY is missing from backend/.env')

client = OpenAI(api_key=api_key)

try:
    completion = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': 'Reply with exactly: API key works.'}],
        temperature=0,
        max_tokens=20,
    )
    print('Success:', completion.choices[0].message.content)
except APIStatusError as error:
    print(f'API request failed: HTTP {error.status_code}')
    print(error.message)
    print('401 = key/authentication problem; 429 = key authenticated but billing/rate-limit problem.')
except Exception as error:
    print(f'Unexpected error: {type(error).__name__}: {error}')
